In [ ]:
import cartopy.crs as ccrs
import numpy as np
#%matplotlib nbagg
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from astropy.wcs import WCS
from astropy.coordinates import ICRS    
from astropy.coordinates import Galactic 
from astropy.coordinates import SkyCoord 
from mpl_toolkits.axes_grid1 import make_axes_locatable
from reproject import reproject_interp
from reproject import reproject_from_healpix
import healpy as hp
from matplotlib import cm
from reproject import reproject_to_healpix
from astropy_healpix import HEALPix
from astropy.io import fits

## Read in GMIMS DRAO (galactic)

In [ ]:
hdu_drao = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_FD_PI.fits')
hdr_drao = hdu_drao[0].header
data_drao = hdu_drao[0].data
print(data_drao.shape)

FD_drao = WCS(hdr_drao).all_pix2world(0,0,range(data_drao.shape[0]),0)[2]
print(FD_drao[0],FD_drao[-1])

## Read in STAPS (equatorial)

In [ ]:
hdu_staps = fits.open('/srv/data/staps/Jan2024/P.cln.fits')
hdr_staps = hdu_staps[0].header
data_staps = hdu_staps[0].data
print(data_staps.shape)

FD_staps = WCS(hdr_staps).all_pix2world(0,0,range(data_staps.shape[0]),0)[2]
print(FD_staps[0],FD_staps[-1])
#print(repr(hdr_staps))

# Make new 2D equatorial and galactic headers

In [ ]:
hdu = fits.PrimaryHDU()
hdr_eq = hdu.header
del hdr_eq['EXTEND']
hdr_eq['BITPIX']   = -32
hdr_eq['NAXIS']    = 2
hdr_eq['NAXIS1']   = 2880
hdr_eq['NAXIS2']   = 1440
hdr_eq['CUNIT1']   = 'deg     '
hdr_eq['CUNIT2']   = 'deg     '
hdr_eq['COORDSYS'] = 'icrs    '
hdr_eq['CTYPE1']   = 'RA---CAR'
hdr_eq['CRPIX1']   = 1440.5
hdr_eq['CRVAL1']   = 180.
hdr_eq['CDELT1']   = -0.125
hdr_eq['CTYPE2']   = 'DEC--CAR'
hdr_eq['CRPIX2']   = 720.5 
hdr_eq['CRVAL2']   = 0.
hdr_eq['CDELT2']   = 0.125
print(repr(hdr_eq))

In [ ]:
hdu = fits.PrimaryHDU()
hdr_gal = hdu.header
del hdr_gal['EXTEND']
hdr_gal['BITPIX']   = -32
hdr_gal['NAXIS']    = 2
hdr_gal['NAXIS1']   = 2880
hdr_gal['NAXIS2']   = 1440
hdr_gal['CUNIT1']   = 'deg     '
hdr_gal['CUNIT2']   = 'deg     '
hdr_gal['COORDSYS'] = 'icrs    '
hdr_gal['CTYPE1']   = 'GLON-CAR'
hdr_gal['CRPIX1']   = 1440.5
hdr_gal['CRVAL1']   = 180.
hdr_gal['CDELT1']   = -0.125
hdr_gal['CTYPE2']   = 'GLAT-CAR'
hdr_gal['CRPIX2']   = 720.5 
hdr_gal['CRVAL2']   = 0.
hdr_gal['CDELT2']   = 0.125
print(repr(hdr_gal))

In [ ]:
def FD_moments(data,peakthresh,maxFD,RM_arr,M0=True,M1=True,M2=False,*args,**kwargs):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use = data[(abs(RM_arr) <= maxFD),:,:]    
    RM_arr_use = RM_arr[(abs(RM_arr) <= maxFD)]  

    # set any data points below PI threshold to NaN:
    data_use[data_use < peakthresh] = np.nan

    moments = {}
    if M0:
        M0_data = dFD*np.nansum(data_use,axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        M1_data = dFD*np.nansum(data_use*RM_arr_use[:,np.newaxis,np.newaxis],axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use[:,np.newaxis,np.newaxis]-M1_data)**2,axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

## Calculate moments from original files

In [ ]:
drao_m = FD_moments(data_drao,0,400,FD_drao)
staps_m = FD_moments(data_staps,0,400,FD_staps)

## Extract M1 and check maps:

In [ ]:
drao_m1_gal = drao_m['M1']
staps_m1_eq = staps_m['M1']

fig = plt.figure(figsize=(12,10))
ax1 = fig.add_subplot(211, projection=WCS(hdr_drao).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_staps).celestial)

ax1.imshow(drao_m1_gal,origin='lower',vmin=-50,vmax=50,cmap='RdBu_r')
ax2.imshow(staps_m1_eq,origin='lower',vmin=-50,vmax=50,cmap='RdBu_r')

## Make 2D versions of original headers

In [ ]:
hdr2D_drao = hdr_drao.copy()
del hdr2D_drao['NAXIS3']
del hdr2D_drao['CRPIX3']
del hdr2D_drao['CDELT3']
del hdr2D_drao['CUNIT3']
del hdr2D_drao['CTYPE3']
del hdr2D_drao['CRVAL3']
hdr2D_drao['NAXIS'] = 2
hdr2D_drao['WCSAXES'] = 2
#print(repr(hdr2D_drao))

hdr2D_staps = hdr_staps.copy()
del hdr2D_staps['NAXIS3']
del hdr2D_staps['CRPIX3']
del hdr2D_staps['CDELT3']
del hdr2D_staps['CTYPE3']
del hdr2D_staps['CRVAL3']
hdr2D_staps['NAXIS'] = 2
hdr2D_staps['WCSAXES'] = 2
#print(repr(hdr2D_staps))

## Make healpix maps of original M1

In [ ]:
drao_m1_gal_hpx, footprint = reproject_to_healpix((drao_m1_gal, hdr2D_drao),'galactic', nside=512)
staps_m1_eq_hpx, footprint = reproject_to_healpix((staps_m1_eq, hdr2D_staps),'ICRS', nside=512)

## Rotate DRAO to EQ and STAPS to GAL

In [ ]:
r1 = hp.Rotator(coord=['C', 'G'])
staps_m1_gal_hpx = np.array(r1.rotate_map_pixel(staps_m1_eq_hpx))

r2 = hp.Rotator(coord=['G', 'C'])
drao_m1_eq_hpx = np.array(r2.rotate_map_pixel(drao_m1_gal_hpx))

## Reproject from healpix to new, matched headers

In [ ]:
staps_m1_gal, footprint = reproject_from_healpix((staps_m1_gal_hpx,'galactic'),hdr_gal, nested=False)
drao_m1_gal,  footprint = reproject_from_healpix((drao_m1_gal_hpx, 'galactic'),hdr_gal, nested=False)

staps_m1_eq, footprint = reproject_from_healpix((staps_m1_eq_hpx,'ICRS'),hdr_eq, nested=False)
drao_m1_eq,  footprint = reproject_from_healpix((drao_m1_eq_hpx, 'ICRS'),hdr_eq, nested=False)

## Fix GMIMS artifact

In [ ]:
#print(drao_m1_gal[1115,1441])
#print(drao_m1_gal[1115,1438])

for i in range(0,drao_m1_gal.shape[0]):
    drao_m1_gal[i,1438:1442] = np.nanmean(drao_m1_gal[i,1437:1442])
    
#print(drao_m1_gal[1173,1442])
#print(drao_m1_gal[1173,1437])

#print(drao_m1_gal[1117,1442])
#print(drao_m1_gal[1117,1437])

for i in range(1117,1174):
    drao_m1_gal[i,1437:1443] = np.nanmean(drao_m1_gal[i,1436:1444])
    
#print(drao_m1_gal[266,1442])
#print(drao_m1_gal[266,1437])

#print(drao_m1_gal[322,1442])
#print(drao_m1_gal[322,1437])

for i in range(266,323):
    drao_m1_gal[i,1437:1443] = np.nanmean(drao_m1_gal[i,1436:1444])

In [ ]:
fig = plt.figure(figsize=(12,12))
ax1 = fig.add_subplot(211, projection=WCS(hdr_gal).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_gal).celestial)
ax1.imshow(drao_m1_gal,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r')
ax2.imshow(staps_m1_gal,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r')

In [ ]:
drao_m1_gal_hpx, footprint = reproject_to_healpix((drao_m1_gal, hdr_gal),'galactic', nside=512)
r2 = hp.Rotator(coord=['G', 'C'])
drao_m1_eq_hpx = np.array(r2.rotate_map_pixel(drao_m1_gal_hpx))
drao_m1_eq,  footprint = reproject_from_healpix((drao_m1_eq_hpx, 'ICRS'),hdr_eq, nested=False)

## Equatorial maps:

In [ ]:
drao_m1_eq2 =drao_m1_eq.copy()
for i in range(0,drao_m1_eq2.shape[0]):
    print(i,drao_m1_eq2[i,0],staps_m1_eq[i,0])

In [ ]:
drao_m1_eq2[0:713] = np.nan
for i in range(0,drao_m1_eq2.shape[0]):
    print(i,drao_m1_eq2[i,0],staps_m1_eq[i,0])

In [ ]:
fig = plt.figure(figsize=(12,12))
ax1 = fig.add_subplot(211, projection=WCS(hdr_eq).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_eq).celestial)
ax1.imshow(drao_m1_eq2,origin='lower',vmin=-50,vmax=50,cmap='RdBu_r')
ax2.imshow(staps_m1_eq,origin='lower',vmin=-50,vmax=50,cmap='RdBu_r')

In [ ]:
fig = plt.figure(figsize=(20,12))
ax1 = fig.add_subplot(111, projection=WCS(hdr_eq).celestial)
ax1.imshow(drao_m1_eq2,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r')
ax1.imshow(staps_m1_eq,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r')

In [ ]:
drao_m1_eq2[np.isnan(drao_m1_eq2)] = 0.0
staps_m1_eq[np.isnan(staps_m1_eq)] = 0.0

In [ ]:
both_eq = drao_m1_eq2+staps_m1_eq
print(both_eq.shape)

In [ ]:
both_eq_hpx, footprint = reproject_to_healpix((both_eq, hdr_eq),'ICRS', nside=512)
r2 = hp.Rotator(coord=['C', 'G'])
both_gal_hpx = np.array(r2.rotate_map_pixel(both_eq_hpx))
both_gal,  footprint = reproject_from_healpix((both_gal_hpx, 'galactic'),hdr_gal, nested=False)

In [ ]:
fig = plt.figure(figsize=(12,12))
ax1 = fig.add_subplot(211, projection=WCS(hdr_eq).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_gal).celestial)
ax1.imshow(both_eq,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r',interpolation='none')
ax2.imshow(both_gal,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r',interpolation='none')
plt.savefig('../gmims_map.pdf')

In [ ]:
hp.orthview(both_eq_hpx,min=-80,max=80,cmap='RdBu_r',rot=(0,90))
hp.graticule()

In [ ]:
hp.orthview(both_gal_hpx,min=-80,max=80,cmap='RdBu_r',rot=(0,90))
hp.graticule()

In [ ]:
fig = plt.figure()
hp.orthview(data,rot=(125,25),half_sky=True,cmap=cmap,min=vmin,max=vmax,
            xsize=1200,cbar=False,fig=fig,badcolor=backgrdclr)
axes = fig.get_axes()
ax = axes[0]
im = ax.get_images()[0]

    cbar = plt.colorbar(im,orientation='horizontal', fraction=0.035, pad=0)
    cbar.set_label(cbartitle, fontsize=fs)
    cbar.ax.tick_params(labelsize=fs, colors='k',grid_color='k')
    
    lon_values = [0,30,60,90,120,150,180,210,240,270,300,330]
    lat_values = [-30,0,30,60,90]
    for lon in lon_values:
        hp.projplot([lon] * 1000, np.linspace(-90, 90, 1000), lonlat=True, color=lineclr, linewidth=0.5)
        if ((lon>30) & (lon<210)):
            hp.projtext(lon, 0, f'{lon}°', lonlat=True, fontsize=fs, ha='center', va='top',color=labelclr)

    for lat in lat_values:
        hp.projplot(np.linspace(0, 360, 1000), [lat] * 1000, lonlat=True, color=lineclr, linewidth=0.5)
        hp.projtext(60, lat, f'{lat}°', lonlat=True, fontsize=fs, ha='right', va='bottom',color=labelclr)
    
    if loop:
        loops(color=loopcolor)    
    
    plt.title(maptitle, fontsize=fs+2)

    plt.savefig(fileout, dpi=600, bbox_inches='tight')